# 🎙️➡️🎨 From Voice to Vision — Fase 0: Setup & Dati

Questo notebook prepara l'ambiente su **Google Colab**, scarica il dataset **RAVDESS**
e fa i primi controlli (distribuzione delle classi, forma d'onda e spettrogramma log-Mel di una clip).

**Come si usa:**
1. Runtime → *Change runtime type* → **T4 GPU** (utile per le fasi successive; la Fase 0 gira anche su CPU).
2. Inserisci l'URL della tua repo GitHub nella cella qui sotto ed esegui le celle in ordine.

In [ ]:
# === 1. Clona la tua repo GitHub ===
# Sostituisci con l'URL della TUA repo (dopo averla creata su GitHub).
REPO_URL = "https://github.com/<tuo-username>/from-voice-to-vision.git"

import os
repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.exists(repo_name):
    !git clone $REPO_URL
%cd $repo_name
!pwd

In [ ]:
# === 2. Installa le librerie audio mancanti su Colab ===
!pip install -q librosa soundfile noisereduce
print("✓ librerie pronte")

In [ ]:
# === 3. Scarica ed estrai RAVDESS (~200 MB, richiede 1-2 min) ===
from src import config, data_loader
data_loader.download_ravdess()

In [ ]:
# === 4. Costruisci l'indice del dataset (metadati per ogni clip) ===
import pandas as pd
df = data_loader.build_index()
print("Totale clip:", len(df))
df.head()

In [ ]:
# === 5. Controlli: split speaker-independent e bilanciamento classi ===
print("Clip per split:")
print(df["split"].value_counts(), "\n")
print("Attori per split (NON devono sovrapporsi):")
for s in ["train", "val", "test"]:
    print(f"  {s:5s}:", sorted(df[df.split==s].actor.unique()))
print("\nClip per emozione:")
print(df["emotion"].value_counts())

In [ ]:
# === 6. Distribuzione delle emozioni ===
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(9,4))
sns.countplot(data=df, x="emotion", order=config.EMOTIONS, hue="split")
plt.title("Distribuzione delle emozioni per split (RAVDESS)")
plt.xticks(rotation=30); plt.ylabel("n. clip"); plt.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "00_emotion_distribution.png", dpi=120)
plt.show()

In [ ]:
# === 7. Ascolta una clip + forma d'onda + spettrogramma log-Mel ===
import librosa, librosa.display, numpy as np
from IPython.display import Audio, display

sample = df[df.emotion == "angry"].iloc[0]
y, sr = librosa.load(sample.path, sr=config.SAMPLE_RATE)
print(f"Emozione: {sample.emotion} | attore: {sample.actor} ({sample.gender}) | durata: {len(y)/sr:.2f}s")
display(Audio(y, rate=sr))

fig, ax = plt.subplots(1, 2, figsize=(13,4))
librosa.display.waveshow(y, sr=sr, ax=ax[0]); ax[0].set_title("Forma d'onda")
S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=config.N_MELS,
                                   n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
S_db = librosa.power_to_db(S, ref=np.max)
img = librosa.display.specshow(S_db, sr=sr, hop_length=config.HOP_LENGTH,
                               x_axis="time", y_axis="mel", ax=ax[1])
ax[1].set_title("Spettrogramma log-Mel"); fig.colorbar(img, ax=ax[1], format="%+2.0f dB")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "00_sample_angry.png", dpi=120); plt.show()

### ✅ Fase 0 completata
Se vedi la distribuzione delle classi, senti l'audio e vedi lo spettrogramma, l'ambiente è pronto.

**Mandami:** l'output della cella 5 (split + conteggi) e le due figure. Poi passiamo alla **Fase 1** (preprocessing + feature).